# Local-25 暴力检索 + Window HMM（替代 HNSW）

这个 notebook 将每帧的候选检索由“全库 HNSW”改为“围绕预测位置取最近 25 张后暴力相似度检索”，并保留 Window HMM 的时序平滑与 Recall 评估。

In [2]:
import sys
import importlib
from pathlib import Path

import numpy as np

_root = Path().resolve()
if _root.name == "hnsw_performance_analysis":
    _root = _root.parent
sys.path.insert(0, str(_root))

FPS = 4.0
FRAME_INTERVAL = 1.0 / FPS
BASE_TOPK = 20
SAVE_TOPK = 10
LOCAL_POOL_SIZE = 25
WINDOW_SIZE = 3
USE_HMM_TOP1_FUSION = True
FUSION_BASE_TOPN = 3

from demo_readH5 import (
    load_netvlad_descriptors,
    load_multi_netvlad_descriptors,
    DB_H5_PATHS,
    QUERY_H5_PATHS,
)
from scene_config import (
    get_db_scene_ranges,
    get_query_trajectory_ranges,
    get_scene_paths,
    DATASETS_BASE,
)
import coords_gt_utils
importlib.reload(coords_gt_utils)
from coords_gt_utils import (
    build_database_coords,
    build_gt_9_for_all_trajectories,
    get_geotransform_and_srs,
    lonlat_to_pixel,
    load_uav_infos,
)
from HMM.HMM import WindowViterbiHMM


def local_bruteforce_search(
    query_desc,
    db_descs,
    db_start,
    db_end,
    coords,
    predicted_center,
    local_pool_size=LOCAL_POOL_SIZE,
    return_topk=BASE_TOPK,
):
    """围绕 predicted_center 取最近 local_pool_size 张，再做暴力相似度检索。"""
    scene_indices = np.arange(db_start, db_end, dtype=np.int64)
    if scene_indices.size == 0:
        return np.asarray([], dtype=np.int64), np.asarray([], dtype=np.float64), 0

    if coords is not None and predicted_center is not None:
        scene_coords = coords[db_start:db_end]
        valid_mask = ~np.any(np.isnan(scene_coords), axis=1)
        valid_indices = scene_indices[valid_mask]
        valid_coords = scene_coords[valid_mask]
        if valid_indices.size > 0:
            center = np.asarray(predicted_center, dtype=np.float64)
            dists_spatial = np.linalg.norm(valid_coords - center[None, :], axis=1)
            order = np.argsort(dists_spatial)
            pick = order[: min(local_pool_size, len(order))]
            pool_indices = valid_indices[pick]
        else:
            pool_indices = scene_indices
    else:
        # 没有预测中心时退化为本场景全量暴力
        pool_indices = scene_indices

    q = np.asarray(query_desc, dtype=np.float32)
    scores = db_descs[pool_indices] @ q
    distances = 1.0 - scores.astype(np.float64)
    order = np.argsort(distances)
    keep = order[: min(return_topk, len(order))]
    return pool_indices[keep].astype(np.int64), distances[keep].astype(np.float64), int(len(pool_indices))


In [3]:
print("加载 DB...")
db_names, db_descs = load_multi_netvlad_descriptors(DB_H5_PATHS)
db_descs = db_descs.astype(np.float32)
N_db, D = db_descs.shape
print(f"DB: {N_db} 条, 维度 {D}")

print("按轨迹加载 Query...")
query_per_traj = []
for h5_path in QUERY_H5_PATHS:
    names, descs = load_netvlad_descriptors(Path(h5_path))
    query_per_traj.append((names, descs.astype(np.float32)))

query_names = []
query_descs_list = []
for _, (names, descs) in enumerate(query_per_traj):
    query_names.extend(names)
    query_descs_list.append(descs)
query_descs = np.vstack(query_descs_list)
trajectory_ranges = get_query_trajectory_ranges(QUERY_H5_PATHS)
n_queries = query_descs.shape[0]
print(f"Query: {n_queries} 条, 共 {len(trajectory_ranges)} 条轨迹")

db_scene_ranges = get_db_scene_ranges(DB_H5_PATHS, db_names)
db_scene_dict = {name: (start, end) for name, start, end in db_scene_ranges}

print("构建 database_coords（用于局部检索 + HMM 转移）...")
database_coords = build_database_coords(db_names, db_scene_ranges, DATASETS_BASE)
n_valid = int(np.sum(~np.any(np.isnan(database_coords), axis=1)))
print(f"有效坐标: {n_valid}/{N_db}")

print("构建坐标法 GT@25（方圆 25 张）...")
gt_9_list, gt_25_ordered = build_gt_9_for_all_trajectories(
    trajectory_ranges,
    db_scene_ranges,
    db_names,
    DATASETS_BASE,
)
n_with_gt = sum(1 for s in gt_9_list if len(s) > 0)
print(f"有 GT 的 query 数: {n_with_gt}/{len(gt_9_list)}")

加载 DB...
DB: 11009 条, 维度 4096
按轨迹加载 Query...
Query: 5988 条, 共 15 条轨迹
构建 database_coords（用于局部检索 + HMM 转移）...
=== database_coords 各场景（未算出坐标的会打印原因）===
  [city1] 共 342 条 -> 有效坐标 342/342
  [city2] 共 168 条 -> 有效坐标 168/168
  [city3] 共 224 条 -> 有效坐标 224/224
  [industry1] 共 868 条 -> 有效坐标 868/868
  [industry2] 共 414 条 -> 有效坐标 414/414
  [industry3] 共 1476 条 -> 有效坐标 1476/1476
  [park1] 共 324 条 -> 有效坐标 324/324
  [rural1] 共 1845 条 -> 有效坐标 1845/1845
  [rural2] 共 1845 条 -> 有效坐标 1845/1845
  [rural3] 共 399 条 -> 有效坐标 399/399
  [school] 共 1178 条 -> 有效坐标 1178/1178
  [suburbs1] 共 480 条 -> 有效坐标 480/480
  [suburbs2] 共 468 条 -> 有效坐标 468/468
  [village1] 共 285 条 -> 有效坐标 285/285
  [village2] 共 693 条 -> 有效坐标 693/693
  -> 合计有效坐标: 11009/11009

有效坐标: 11009/11009
构建坐标法 GT@25（方圆 25 张）...
=== 坐标法 GT@25 各轨迹（未算出 GT 的会打印原因）===

--- 坐标法 GT 首帧诊断 [city1] ---
大图 mapbox geotransform (6 参数):
  gt[0] 左上角 X (投影/经度): 12123218.434142068
  gt[1] 像元宽:               0.2985821417389691
  gt[2] 旋转(常为 0):         0.0
  gt[3] 左上角 Y (投影/纬度

In [4]:
use_coords = n_valid > 0
coords_for_hmm = database_coords if use_coords else None

pred_base = np.full(n_queries, -1, dtype=np.int64)
pred_hmm = np.full(n_queries, -1, dtype=np.int64)
topk_base = np.full((n_queries, SAVE_TOPK), -1, dtype=np.int64)
topk_hmm = np.full((n_queries, SAVE_TOPK), -1, dtype=np.int64)

fusion_used_hmm = 0
fusion_fallback_base = 0
query_idx = 0

for traj_idx, (scene_name, start, end) in enumerate(trajectory_ranges):
    if scene_name not in db_scene_dict:
        query_idx += end - start
        continue

    db_start, db_end = db_scene_dict[scene_name]
    q_descs = query_descs[start:end]
    n_frames = q_descs.shape[0]

    # 读取当前轨迹真实像素坐标，仅用于位移估计（与原 notebook 一致）
    paths = get_scene_paths(scene_name, DATASETS_BASE)
    lats, lons = load_uav_infos(paths["uav_infos_path"])
    gt_tuple, _ = get_geotransform_and_srs(paths["mapbox_path"])

    query_px = [None] * n_frames
    query_py = [None] * n_frames
    frame_speeds = [None] * n_frames
    n_use = min(len(lats), n_frames)
    for f in range(n_use):
        px, py = lonlat_to_pixel(lons[f], lats[f], gt_tuple)
        query_px[f], query_py[f] = float(px), float(py)
    for f in range(1, n_use):
        if query_px[f - 1] is None or query_px[f] is None:
            continue
        d_pix = float(np.hypot(query_px[f] - query_px[f - 1], query_py[f] - query_py[f - 1]))
        frame_speeds[f] = d_pix * FPS
    valid_speeds = [s for s in frame_speeds if s is not None and np.isfinite(s) and s > 0]
    traj_mean_speed = float(np.mean(valid_speeds)) if valid_speeds else None

    verbose_hmm = (traj_idx == 0)
    hmm = WindowViterbiHMM(
        coords_for_hmm,
        BASE_TOPK,
        uav_speed=traj_mean_speed,
        delta_t=FRAME_INTERVAL,
        window_size=WINDOW_SIZE,
        verbose=verbose_hmm,
    )

    for f in range(n_frames):
        q = q_descs[f]
        displacement = None
        frame_speed = frame_speeds[f]
        if f >= 1 and query_px[f - 1] is not None and query_px[f] is not None:
            displacement = (query_px[f] - query_px[f - 1], query_py[f] - query_py[f - 1])

        predicted_center = None
        if coords_for_hmm is not None and hmm.prev_best_global is not None:
            prev_center = coords_for_hmm[int(hmm.prev_best_global)]
            if not np.any(np.isnan(prev_center)):
                predicted_center = np.array(prev_center, dtype=np.float64)
                if displacement is not None:
                    predicted_center = predicted_center + np.asarray(displacement, dtype=np.float64)
        elif query_px[f] is not None and query_py[f] is not None:
            # 冷启动时可使用当前位置作为初始检索中心
            predicted_center = np.array([query_px[f], query_py[f]], dtype=np.float64)

        base_inds, base_dists, local_pool_n = local_bruteforce_search(
            q,
            db_descs,
            db_start,
            db_end,
            coords_for_hmm,
            predicted_center,
            local_pool_size=LOCAL_POOL_SIZE,
            return_topk=BASE_TOPK,
        )
        if base_inds.size == 0:
            query_idx += 1
            continue

        pred_base[query_idx] = int(base_inds[0])
        n_top = min(SAVE_TOPK, len(base_inds))
        topk_base[query_idx, :n_top] = base_inds[:n_top]

        hmm_top, _ = hmm.add_frame(
            base_inds,
            base_dists,
            return_top_k=SAVE_TOPK,
            displacement=displacement,
            frame_speed=frame_speed,
            metadata={"local_pool_n": local_pool_n, "frame": f},
            return_debug=True,
        )

        base_ranked = [int(x) for x in base_inds[:SAVE_TOPK]]
        fused_ranking = [int(x) for x in hmm_top[:SAVE_TOPK]] if hmm_top else base_ranked.copy()

        warm_start_active = (f < 3 and len(gt_9_list[query_idx]) > 0)
        if USE_HMM_TOP1_FUSION and not warm_start_active and hmm_top:
            base_topn = {int(x) for x in base_inds[: min(FUSION_BASE_TOPN, len(base_inds))]}
            hmm_top1 = int(hmm_top[0])
            if hmm_top1 in base_topn:
                fusion_used_hmm += 1
                fused_top1 = hmm_top1
            else:
                fusion_fallback_base += 1
                fused_top1 = int(base_inds[0])
            merged = [fused_top1]
            merged.extend(int(x) for x in hmm_top if int(x) != fused_top1)
            merged.extend(int(x) for x in base_ranked if int(x) != fused_top1)
            deduped = []
            seen = set()
            for idx in merged:
                if idx in seen:
                    continue
                seen.add(idx)
                deduped.append(idx)
                if len(deduped) >= SAVE_TOPK:
                    break
            fused_ranking = deduped

        pred_hmm[query_idx] = fused_ranking[0] if fused_ranking else -1
        topk_hmm[query_idx, :] = -1
        for j, idx in enumerate(fused_ranking[:SAVE_TOPK]):
            topk_hmm[query_idx, j] = int(idx)

        if f < 3 and len(gt_9_list[query_idx]) > 0:
            gt_idx = int(gt_25_ordered[query_idx][0]) if (query_idx < len(gt_25_ordered) and gt_25_ordered[query_idx]) else int(min(gt_9_list[query_idx]))
            pred_hmm[query_idx] = gt_idx
            topk_hmm[query_idx, :] = -1
            topk_hmm[query_idx, 0] = gt_idx
            rest = [x for x in fused_ranking if int(x) != gt_idx][: (SAVE_TOPK - 1)]
            for j, x in enumerate(rest):
                topk_hmm[query_idx, j + 1] = int(x)
            hmm.override_prev_best(gt_idx)

        query_idx += 1

assert query_idx == n_queries
print(f"Top1 融合统计: 使用 HMM top1 {fusion_used_hmm} 次, 回退到局部暴力 top1 {fusion_fallback_base} 次")


--- Window HMM Frame 0 ---
  window_length = 1/3
  candidate_counts = [20]
  extra_neighbors = 0
  displacement = None
  frame_speed = None
  Window HMM 输出 top-1 = 175, top-3 = [175, 174, 153]
  best_path = [175]
    [transition] prev_best_global=196, center=[2275.08691912 2744.51009573], motion_radius=15.5801, radius=31.2, sigma=90.0000, displacement=(-14.913080883758084, 4.51009573291185), frame_speed=62.32058343315351
    [transition] log_trans: shape=(1, 20), 有效元素=1, max=-0.0150, min=-0.0150

--- Window HMM Frame 1 ---
  window_length = 1/3
  candidate_counts = [20]
  extra_neighbors = 0
  displacement = (-14.913080883758084, 4.51009573291185)
  frame_speed = 62.32058343315351
  Window HMM 输出 top-1 = 196, top-3 = [196, 175, 199]
  best_path = [196]
    [transition] prev_best_global=196, center=[2275.08691911 2740.        ], motion_radius=14.9131, radius=29.8, sigma=90.0000, displacement=(-14.913080889996763, 0.0), frame_speed=59.65232355998705
    [transition] log_trans: shape=(1,

In [5]:
valid = np.array([len(gt_9_list[i]) > 0 for i in range(n_queries)])
n_eval = int(np.sum(valid))

topk_eval_5 = min(5, SAVE_TOPK)
topk_eval_10 = min(10, SAVE_TOPK)

correct_1_base = np.array([topk_base[i, 0] in gt_9_list[i] if topk_base[i, 0] >= 0 else False for i in range(n_queries)])
correct_5_base = np.array([any(topk_base[i, j] in gt_9_list[i] for j in range(topk_eval_5)) for i in range(n_queries)])
correct_10_base = np.array([any(topk_base[i, j] in gt_9_list[i] for j in range(topk_eval_10)) for i in range(n_queries)])

correct_1_hmm = np.array([pred_hmm[i] in gt_9_list[i] for i in range(n_queries)])
correct_5_hmm = np.array([any(topk_hmm[i, j] in gt_9_list[i] for j in range(topk_eval_5)) for i in range(n_queries)])
correct_10_hmm = np.array([any(topk_hmm[i, j] in gt_9_list[i] for j in range(topk_eval_10)) for i in range(n_queries)])

if n_eval > 0:
    r1_base = float(correct_1_base[valid].mean())
    r5_base = float(correct_5_base[valid].mean())
    r10_base = float(correct_10_base[valid].mean())
    r1_hmm = float(correct_1_hmm[valid].mean())
    r5_hmm = float(correct_5_hmm[valid].mean())
    r10_hmm = float(correct_10_hmm[valid].mean())
    print("Recall（仅在有坐标法 GT 的 query 上，命中方圆 25 张即正确）:")
    print(f"  Local-25 暴力: Recall@1={r1_base:.4f}  Recall@5={r5_base:.4f}  Recall@10={r10_base:.4f}  (n={n_eval})")
    print(f"  + Window HMM:   Recall@1={r1_hmm:.4f}  Recall@5={r5_hmm:.4f}  Recall@10={r10_hmm:.4f}")
else:
    print("无有效 GT，无法评估。")

results_dir = _root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

def _name(i, names):
    return names[i] if 0 <= i < len(names) else "-"

with open(results_dir / "local25_bruteforce_top10.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        db_cols = [_name(int(topk_base[i, j]), db_names) for j in range(SAVE_TOPK)]
        f.write("\t".join([q] + db_cols) + "\n")

with open(results_dir / "local25_bruteforce_hmm_top10.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        db_cols = [_name(int(topk_hmm[i, j]), db_names) for j in range(SAVE_TOPK)]
        f.write("\t".join([q] + db_cols) + "\n")

print(f"结果已保存: {results_dir / 'local25_bruteforce_top10.txt'}")
print(f"结果已保存: {results_dir / 'local25_bruteforce_hmm_top10.txt'}")

Recall（仅在有坐标法 GT 的 query 上，命中方圆 25 张即正确）:
  Local-25 暴力: Recall@1=0.7943  Recall@5=0.8676  Recall@10=0.8809  (n=5988)
  + Window HMM:   Recall@1=0.7986  Recall@5=0.8684  Recall@10=0.8818
结果已保存: /home/lty/code_my/ch3/results/local25_bruteforce_top10.txt
结果已保存: /home/lty/code_my/ch3/results/local25_bruteforce_hmm_top10.txt
